In [10]:
import csv
import random
from faker import Faker

# Inicializamos Faker con múltiples configuraciones regionales
# La de España (es_ES) la dejamos como principal para los apellidos y el DNI
fake_es = Faker("es_ES")

# Creamos una lista de generadores para los nombres extranjeros
locales_extranjeros = [
    Faker("en_US"),  # Inglés
    Faker("fr_FR"),  # Francés
    Faker("it_IT"),  # Italiano
    Faker("de_DE"),  # Alemán
]


def generar_dni_valido():
    letras = "TRWAGMYFPDXBNJZSQVHLCKE"
    numero = random.randint(10000000, 99999999)
    letra = letras[numero % 23]
    return f"{numero}{letra}"


usuarios = []

for i in range(1, 1001):
    # Decidimos un 70% de nombres españoles y un 30% de extranjeros (puedes cambiar este ratio)
    if random.random() > 0.30:
        nombre = fake_es.first_name()
    else:
        # Elige un Faker extranjero al azar y genera el nombre
        fake_extranjero = random.choice(locales_extranjeros)
        nombre = fake_extranjero.first_name()

    # Mantenemos los apellidos y el DNI con el formato local para simular residentes
    apellido = fake_es.last_name() + " " + fake_es.last_name()
    dni = generar_dni_valido()

    # Email limpio basado en las variables anteriores
    primer_apellido = apellido.split()[0].lower()
    email = f"{nombre.lower()}.{primer_apellido}{i}@example.com"

    # Limpieza de caracteres conflictivos para entornos de desarrollo/DB
    email = (
        email.replace("ñ", "n")
        .replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
        .replace("ç", "c")
        .replace("ü", "u")
        .replace("ä", "a")
        .replace("ö", "o")
        .replace("ß", "ss")
    )

    usuarios.append(
        {
            "id": i,
            "nombre": nombre,
            "apellido": apellido,
            "dni": dni,
            "email": email,
        }
    )

# Guardar en CSV
with open("../data/raw/usuarios_mixtos.csv", mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=["id", "nombre", "apellido", "dni", "email"]
    )
    writer.writeheader()
    writer.writerows(usuarios)

print(
    f"¡Hecho! Generados {len(usuarios)} registros con un mix de nombres internacionales."
)

¡Hecho! Generados 1000 registros con un mix de nombres internacionales.


In [ ]:
import pandas as pd

# 1. Asumiendo que ya tienes tu DataFrame original con los 1000 usuarios
# df_usuarios = ... (el dataframe generado con Faker)

# 2. Cargamos el nuevo CSV de fraude y riesgo país
ruta_fraude = "../data/raw/dataset_fraude_riesgo_pais_ult.csv"
df_usuarios = pd.read_csv("../data/raw/usuarios_mixtos.csv")  # Cargamos el DataFrame de usuarios
df_fraude = pd.read_csv(ruta_fraude)

# 3. Columnas específicas que necesitamos extraer + la clave de unión
columnas_interes = [
    "id_usuario",
    "email_verificado",
    "dias_antiguedad_cuenta",
    "pais_emision",
    "paso_3d_secure",
]

# Filtramos el dataframe de fraude para traer solo las columnas requeridas
df_fraude_filtrado = df_fraude[columnas_interes]

# 4. Hacemos el merge relacionando 'id' (de tu df) con 'id_usuario' (del nuevo csv)
df_final = df_usuarios.merge(
    df_fraude_filtrado, left_on="id", right_on="id_usuario", how="left"
)

# 5. Opcional: Como ya tienes la columna 'id', la columna 'id_usuario' queda duplicada.
# Podemos eliminarla para dejar el DataFrame limpio.
df_final = df_final.drop(columns=["id_usuario"])

# Verificamos el resultado
print("Columnas resultantes:", df_final.columns.tolist())
print(df_final.head())

Columnas resultantes: ['id', 'nombre', 'apellido', 'dni', 'email', 'email_verificado', 'dias_antiguedad_cuenta', 'pais_emision', 'paso_3d_secure']
   id nombre        apellido        dni                        email  \
0   1  Maite  Figueroa Barco  49587254J  maite.figueroa1@example.com   
1   1  Maite  Figueroa Barco  49587254J  maite.figueroa1@example.com   
2   1  Maite  Figueroa Barco  49587254J  maite.figueroa1@example.com   
3   1  Maite  Figueroa Barco  49587254J  maite.figueroa1@example.com   
4   1  Maite  Figueroa Barco  49587254J  maite.figueroa1@example.com   

   email_verificado  dias_antiguedad_cuenta pais_emision  paso_3d_secure  
0                 1                     370           RU               0  
1                 1                     370           ES               0  
2                 1                     370           ES               0  
3                 1                     370           FR               0  
4                 1                     370  

In [12]:
df_final

,id,nombre,apellido,dni,email,email_verificado,dias_antiguedad_cuenta,pais_emision,paso_3d_secure
0,1,Maite,Figueroa Barco,49587254J,maite.figueroa1@example.com,1,370,RU,0
1,1,Maite,Figueroa Barco,49587254J,maite.figueroa1@example.com,1,370,ES,0
2,1,Maite,Figueroa Barco,49587254J,maite.figueroa1@example.com,1,370,ES,0
3,1,Maite,Figueroa Barco,49587254J,maite.figueroa1@example.com,1,370,FR,0
4,1,Maite,Figueroa Barco,49587254J,maite.figueroa1@example.com,1,370,US,1
...,...,...,...,...,...,...,...,...,...
10058,1000,Armando,Gallo Gargallo,27640405V,armando.gallo1000@example.com,1,1282,RU,0
10059,1000,Armando,Gallo Gargallo,27640405V,armando.gallo1000@example.com,1,1282,RU,0
10060,1000,Armando,Gallo Gargallo,27640405V,armando.gallo1000@example.com,1,1282,RU,1
10061,1000,Armando,Gallo Gargallo,27640405V,armando.gallo1000@example.com,1,1282,ES,0


In [24]:
import csv
import random
from faker import Faker
import pandas as pd

# 1. Cargar el DataFrame de fraude y extraer los id_usuario únicos
ruta_fraude = "../data/raw/dataset_fraude_riesgo_pais_ult.csv"
df_fraude = pd.read_csv(ruta_fraude)

# Extraemos los id_usuario únicos directamente usando el nombre exacto
ids_unicos = df_fraude['id_usuario'].unique()

# Inicializamos Faker con las configuraciones regionales
fake_es = Faker("es_ES")
locales_extranjeros = [
    Faker("en_US"),  # Inglés
    Faker("fr_FR"),  # Francés
    Faker("it_IT"),  # Italiano
    Faker("de_DE"),  # Alemán
]

def generar_dni_valido():
    letras = "TRWAGMYFPDXBNJZSQVHLCKE"
    numero = random.randint(10000000, 99999999)
    letra = letras[numero % 23]
    return f"{numero}{letra}"

usuarios = []

# 2. Generar un perfil por cada id_usuario del dataset de fraude
for id_usuario in ids_unicos:
    if random.random() > 0.30:
        nombre = fake_es.first_name()
    else:
        fake_extranjero = random.choice(locales_extranjeros)
        nombre = fake_extranjero.first_name()

    apellido = fake_es.last_name() + " " + fake_es.last_name()
    dni = generar_dni_valido()

    # Email limpio basado en el id_usuario
    primer_apellido = apellido.split()[0].lower()
    email = f"{nombre.lower()}.{primer_apellido}@example.com"

    # Limpieza de caracteres conflictivos
    email = (
        email.replace("ñ", "n")
        .replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
        .replace("ç", "c")
        .replace("ü", "u")
        .replace("ä", "a")
        .replace("ö", "o")
        .replace("ß", "ss")
    )

    usuarios.append(
        {
            "id_usuario": id_usuario,  # Clave renombrada a id_usuario
            "nombre": nombre,
            "apellido": apellido,
            "dni": dni,
            "email": email,
        }
    )

# 3. Guardar en CSV con la cabecera id_usuario
with open("../data/raw/usuarios_mixtos.csv", mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=["id_usuario", "nombre", "apellido", "dni", "email"]
    )
    writer.writeheader()
    writer.writerows(usuarios)

print(
    f"¡Hecho! Generados {len(usuarios)} registros emparejados por 'id_usuario'."
)

¡Hecho! Generados 9999 registros emparejados por 'id_usuario'.


In [25]:
import pandas as pd

# 2. Cargamos los archivos
ruta_fraude = "../data/raw/dataset_fraude_riesgo_pais_ult.csv"
df_usuarios = pd.read_csv("../data/raw/usuarios_mixtos.csv")
df_fraude = pd.read_csv(ruta_fraude)

# 3. Columnas específicas que necesitamos extraer
columnas_interes = [
    "id_usuario",
    "email_verificado",
    "dias_antiguedad_cuenta",
    "pais_emision",
    "paso_3d_secure",
]

# --- NUEVA LÓGICA: Limpieza de duplicados seleccionando al azar ---
# 1. Filtramos las columnas que nos interesan.
# 2. .sample(frac=1, random_state=42) desordena las filas al azar.
# 3. .drop_duplicates(subset=['id_usuario'], keep='first') se queda con la primera fila que encuentra de cada usuario (que ahora es una aleatoria).
df_fraude_unico = (
    df_fraude[columnas_interes]
    .sample(frac=1, random_state=42)
    .drop_duplicates(subset=["id_usuario"], keep="first")
)

# 4. Hacemos el merge con los datos ya limpios (1 fila por usuario)
df_final = df_usuarios.merge(
    df_fraude_unico, left_on="id_usuario", right_on="id_usuario", how="left"
)

# 5. Eliminamos la columna repetida
df_final = df_final.drop(columns=["id_usuario"])

# Verificamos el resultado (debería darte exactamente 1000 filas)
print(f"Total de registros finales: {len(df_final)}")
print("Columnas resultantes:", df_final.columns.tolist())
print(df_final.head())

Total de registros finales: 9999
Columnas resultantes: ['nombre', 'apellido', 'dni', 'email', 'email_verificado', 'dias_antiguedad_cuenta', 'pais_emision', 'paso_3d_secure']
      nombre            apellido        dni                         email  \
0  Ingetraud       Grande Molina  85815202V  ingetraud.grande@example.com   
1    Melinda         Blasco Rosa  89888740W    melinda.blasco@example.com   
2   Hernando        Reguera Losa  66360842T  hernando.reguera@example.com   
3      Yaiza  Aramburu Contreras  75809185N    yaiza.aramburu@example.com   
4     Roldán     Ariza Salamanca  55144736M      roldan.ariza@example.com   

   email_verificado  dias_antiguedad_cuenta pais_emision  paso_3d_secure  
0                 1                    1268           RU               1  
1                 1                     176           ES               1  
2                 1                     101           US               0  
3                 1                     555           US       

In [26]:
import pandas as pd

# 2. Cargamos los archivos
ruta_fraude = "../data/raw/dataset_fraude_riesgo_pais_ult.csv"
df_usuarios = pd.read_csv("../data/raw/usuarios_mixtos.csv")
df_fraude = pd.read_csv(ruta_fraude)

# 3. Columnas específicas que necesitamos extraer
columnas_interes = [
    "id_usuario",
    "email_verificado",
    "dias_antiguedad_cuenta",
    "pais_emision",
    "paso_3d_secure",
]

# --- NUEVA LÓGICA: Limpieza de duplicados seleccionando al azar ---
df_fraude_unico = (
    df_fraude[columnas_interes]
    .sample(frac=1, random_state=42)
    .drop_duplicates(subset=["id_usuario"], keep="first")
)

# 4. Hacemos el merge por la columna 'id_usuario'
# Al usar 'on', Pandas entiende que es la misma columna en ambos lados y no la duplica
df_final = df_usuarios.merge(df_fraude_unico, on="id_usuario", how="left")

# --- ELIMINADO EL DROP ---
# Ya no es necesario eliminarla porque no se genera una columna duplicada (como id_usuario_x / id_usuario_y)

# Verificamos el resultado
print(f"Total de registros finales: {len(df_final)}")
print("Columnas resultantes:", df_final.columns.tolist())
print(df_final.head())

Total de registros finales: 9999
Columnas resultantes: ['id_usuario', 'nombre', 'apellido', 'dni', 'email', 'email_verificado', 'dias_antiguedad_cuenta', 'pais_emision', 'paso_3d_secure']
                             id_usuario     nombre            apellido  \
0  05338752-0e67-41fb-94b6-5dd9394331d1  Ingetraud       Grande Molina   
1  cccb39d2-24ec-4f32-8ee5-06afbccbd45c    Melinda         Blasco Rosa   
2  6408c722-1bff-4c53-b23b-d850dc630650   Hernando        Reguera Losa   
3  24738022-69bd-4f51-9e93-a08a65f27b4f      Yaiza  Aramburu Contreras   
4  d3f8b738-0839-4903-b85b-8f29d6bd1d9c     Roldán     Ariza Salamanca   

         dni                         email  email_verificado  \
0  85815202V  ingetraud.grande@example.com                 1   
1  89888740W    melinda.blasco@example.com                 1   
2  66360842T  hernando.reguera@example.com                 1   
3  75809185N    yaiza.aramburu@example.com                 1   
4  55144736M      roldan.ariza@example.com     

In [27]:
df_final

,id_usuario,nombre,apellido,dni,email,email_verificado,dias_antiguedad_cuenta,pais_emision,paso_3d_secure
0,05338752-0e67-41fb-94b6-5dd9394331d1,Ingetraud,Grande Molina,85815202V,ingetraud.grande@example.com,1,1268,RU,1
1,cccb39d2-24ec-4f32-8ee5-06afbccbd45c,Melinda,Blasco Rosa,89888740W,melinda.blasco@example.com,1,176,ES,1
2,6408c722-1bff-4c53-b23b-d850dc630650,Hernando,Reguera Losa,66360842T,hernando.reguera@example.com,1,101,US,0
3,24738022-69bd-4f51-9e93-a08a65f27b4f,Yaiza,Aramburu Contreras,75809185N,yaiza.aramburu@example.com,1,555,US,0
4,d3f8b738-0839-4903-b85b-8f29d6bd1d9c,Roldán,Ariza Salamanca,55144736M,roldan.ariza@example.com,1,707,DE,1
...,...,...,...,...,...,...,...,...,...
9994,9e5b8974-215e-42d8-8bcb-ae0f8d519642,Vasco,Jerez Borja,15284318J,vasco.jerez@example.com,1,574,US,1
9995,ffb21f31-f7e0-4394-b2d8-1f44c4653e8c,Lara,Bertrán Noriega,90985001Z,lara.bertran@example.com,1,998,ES,0
9996,9e787470-0a10-418d-89ee-1e9ad453248d,Ángela,Solera Abad,11891439W,angela.solera@example.com,0,1065,ES,0
9997,51b91cdb-1def-4b7a-9e3b-2bca3b039a19,Nicole,Guardiola Matas,19075269N,nicole.guardiola@example.com,1,996,FR,0


In [28]:
df_final.to_csv('../data/processed/base_users.csv', index=False)